# Enfoque con Embeddings
Todo a embeddings (queries, sinopsis + año + director + keywords) --> 5 con mas similtud coseno (comparando queries vs sinopsis + año + director + keywords)

1. unificar texto
2. embeddings con w2v o sentence transformer sobre texto y queries
3. similitud coseno text vs queries

In [1]:
!pip install datasets
!pip install sentence-transformers
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 67.3 MB/s eta 0:00:00:00:0100:01


In [2]:
from datasets import load_dataset
import pandas as pd
import re       # libreria de expresiones regulares
import string   # libreria de cadena de caracteres
from gensim.models.phrases import Phrases, Phraser
import multiprocessing
from gensim.models import Word2Vec
import numpy as np

## Carga de datasets

Traemos el dataset de sinopsis de peliculas de IMDb desde Hugging Face

In [3]:
sinopsis = load_dataset("mathigatti/spanish_imdb_synopsis")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

plots.csv:   0%|          | 0.00/1.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4967 [00:00<?, ? examples/s]

Traemos el dataset proporcionado con la información de los usuarios y sus respectivas queries

In [25]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/usuarios/usuarios.csv")

In [26]:
usuarios

,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
0,U01,Valentina,definido,Durmiendo con su enemigo,Más allá de la muerte,Desaparecida,¡Olvídate de mí!,The Crazies,Quiero una película donde una mujer enfrenta u...
1,U02,Rodrigo,definido,Adiós Bafana,Mi pie izquierdo,L.A. Confidential,Érase una vez en América,El juego del halcón,Busco algo basado en hechos reales sobre corru...
2,U03,Camila,definido,Los padres de él,Mamá a la fuerza,Norbit,Elizabethtown,My Sassy Girl,Una comedia donde la relación entre dos person...
3,U04,Tomás,definido,Matrix,Fahrenheit 451,¡Olvídate de mí!,X-Men,El único,Algo que haga pensar sobre qué es real y qué e...
4,U05,Lucía,definido,La novia cadáver,Spirit: El corcel indomable,Las aventuras de Peabody y Sherman,Los Increíbles,Steamboy,Animación donde el protagonista lucha por su l...
5,U06,Martín,definido,Bienvenidos a Collinwood,El gran golpe,L.A. Confidential,Sympathy for Mr. Vengeance,La otra cara del crimen,Un grupo de personas planea un robo o estafa y...
6,U07,Sofía,definido,Velvet Goldmine,"Cuanto más, ¡mejor!",La vida de bohemia,Cero en conducta,Corazón salvaje,Una película sobre músicos o artistas que vive...
7,U08,Diego,definido,Superdetective en Hollywood,Mission: Impossible,Misión: Imposible 3,"Walker, Texas Ranger",300,Acción directa con un héroe que trabaja solo o...
8,U09,Elena,definido,Viaje a Darjeeling,Mi Idaho privado,Melinda y Melinda,La ciencia del sueño,Un beso,Algo tranquilo sobre personas que intentan rec...
9,U10,Facundo,definido,Sátántangó,Corazón salvaje,Mi Idaho privado,La ciencia del sueño,Sympathy for Mr. Vengeance,"Algo que sea difícil de clasificar, con una ló..."


Visualizamos las queries

In [27]:
for texto in usuarios['query']:
    print(texto)

Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Busco algo basado en hechos reales sobre corrupción o poder político
Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
Una película sobre músicos o artistas que viven al margen, con mucha atmósfera y estilo visual
Acción directa con un héroe que trabaja solo o casi solo contra una organización criminal o corrupta
Algo tranquilo sobre personas que intentan reconectar o entenderse después de una distancia larga
Algo que sea difícil de clasificar, con una lógica narrativa propia, no convencional
No sé bien, algo que valga la pena ver un domingo a la noche, que enganche desde el princi

Construimos el dataset de peliculas, agregando un id que faltaba.

In [28]:
df_pelis = pd.DataFrame(sinopsis['train'])
df_pelis["id"] = df_pelis.index + 1
df_pelis.head()

,description,keywords,genre,year,name,director,id
0,"Orin Boyd, un duro policía de una comisaría de...","vietnam war veteran, heroína, drogas, narcotra...","acción, crimen, suspense",2001.0,Herida abierta,Andrzej Bartkowiak,1
1,Al llegar a un pequeño pueblo donde ha heredad...,"herencia, hostess, comedia negra, pueblo, magia","comedia, terror",1989.0,"Elvira, reina de las tinieblas",James Signorelli,2
2,Una mujer finge su muerte en un intento de esc...,"violencia doméstica, muerte fingida, borderlin...","drama, suspense",1991.0,Durmiendo con su enemigo,Joseph Ruben,3
3,Durante un memorial en la ciudad natal de su p...,"manic pixie dream girl, publicidad, bad public...","comedia, drama, romance",2005.0,Elizabethtown,Cameron Crowe,4
4,Las pruebas nucleares francesas irradian a una...,"monstruo gigante, iguana, militar, giant footp...","acción, ciencia ficción, suspense",1998.0,Godzilla,Roland Emmerich,5


## Preprocesado

Defino una funcion para limpiar texto

In [29]:
def limpiar_texto(text):
    # pasa las mayusculas del texto a minusculas
    text = text.lower()
    # reemplaza texto entre corchetes por espacio en blanco
    text = re.sub(r'\[.*?¿\]%', ' ', text)
    # reemplaza signos de puntuacion por espacio en blanco
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    # remueve palabras que contienen numeros.
    text = re.sub(r'\w*\d\w*', '', text)
    # remueve caracteres especiales y saltos de linea
    text = re.sub('[‘’“”…«»]', '', text)
    text = re.sub('\n', ' ', text)
    return text

Unificamos las variables relevantes en un texto

In [30]:
df_pelis["texto"] = (
    df_pelis["name"] + " "
    + df_pelis["description"] + " "
    + df_pelis["year"].fillna('').astype(str) + " "
    + df_pelis["director"].fillna('') + " "
    + df_pelis["genre"] + " "
    + df_pelis["keywords"]
)

In [31]:
limpieza = lambda x: limpiar_texto(x)
data_clean = pd.DataFrame(df_pelis["texto"].apply(limpieza))

Agregamos bigramas al corpus (?)

In [32]:
input = [row.split() for row in data_clean["texto"]] # separamos en una lista
phrases = Phrases(input, min_count=20, progress_per=1000)

bigram = Phraser(phrases)

sentences = bigram[input]

## Embedding de peliculas

### Entrenamos modelo World2Vec

> [!!!] falta elegir los parametros del modelo acorde al trabajo.

In [33]:
cores = multiprocessing.cpu_count()

w2v_model = Word2Vec(min_count=20, # ignora palabras cuya frecuencia es menor a esta
                     window=2, # tamanio de la ventana de contexto
                     vector_size=300, # dimension del embedding
                     sample=6e-5, # umbral para downsamplear palabras muy frecuentes
                     alpha=0.03, # tasa de aprendizaje inicial (entrenamiento de la red neuronal)
                     min_alpha=0.0007, # tasa de aprendizaje minima
                     negative=20, # penalidad de palabras muy frecuentes o poco informaitvas
                     workers=cores) # numero de cores para entrenar el modelo

w2v_model.build_vocab(sentences, progress_per=10000) # construye el vocabulario

### ENTRENA EL MODELO
w2v_model.train(sentences, total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)

(1273468, 6144450)

### Calcular vector promedio de cada película

Definimos una funcion que calcula el vector promedio a partir de un texto

In [34]:
def obtener_vector_promedio(texto, modelo):
    palabras = texto.split()

    vectores_palabras = [modelo.wv[palabra] for palabra in palabras if palabra in modelo.wv]

    if not vectores_palabras:
        return np.zeros(modelo.wv.vector_size)

    return np.mean(vectores_palabras, axis=0)

Calculamos el embedding promedio de cada pelicula

In [35]:
embeddings_peliculas = pd.DataFrame(np.array([obtener_vector_promedio(text, w2v_model) for text in data_clean['texto']]))

In [36]:
embeddings_peliculas.head()

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,0.025147,-0.045438,0.112312,0.174055,-0.098421,0.038913,-0.086071,0.299486,0.039170,-0.002519,...,0.017518,0.079891,0.366359,-0.116127,0.184618,0.052306,0.124427,-0.058356,0.136525,0.045512
1,0.045994,-0.036832,0.117254,0.266986,-0.121041,0.028489,-0.121183,0.274124,-0.017013,-0.000061,...,0.077043,0.151034,0.285719,-0.088248,0.201177,0.083711,0.112930,-0.053855,0.121345,0.034310
2,0.030151,-0.040465,0.058267,0.266482,-0.131634,0.009647,-0.135305,0.274111,-0.010271,-0.055769,...,0.071606,0.158415,0.262805,-0.072939,0.223894,0.081897,0.063673,-0.047841,0.099216,-0.003506
3,0.038538,-0.024892,0.088787,0.257778,-0.110085,0.017308,-0.123375,0.300605,0.000616,-0.032242,...,0.070952,0.141597,0.276420,-0.086910,0.202238,0.088911,0.088554,-0.060159,0.109176,0.032100
4,0.041123,-0.059851,0.130128,0.262925,-0.114963,0.031149,-0.133662,0.248894,-0.001590,0.017231,...,0.066907,0.141212,0.319275,-0.089438,0.203360,0.062950,0.118264,-0.042241,0.128754,0.039183


Estan en orden entonces podemos joinear por index

In [37]:
pelis_embd = embeddings_peliculas.merge(df_pelis[["name","id"]], left_index=True, right_index=True)
pelis_embd.head()

,0,1,2,3,4,5,6,7,8,9,...,292,293,294,295,296,297,298,299,name,id
0,0.025147,-0.045438,0.112312,0.174055,-0.098421,0.038913,-0.086071,0.299486,0.039170,-0.002519,...,0.366359,-0.116127,0.184618,0.052306,0.124427,-0.058356,0.136525,0.045512,Herida abierta,1
1,0.045994,-0.036832,0.117254,0.266986,-0.121041,0.028489,-0.121183,0.274124,-0.017013,-0.000061,...,0.285719,-0.088248,0.201177,0.083711,0.112930,-0.053855,0.121345,0.034310,"Elvira, reina de las tinieblas",2
2,0.030151,-0.040465,0.058267,0.266482,-0.131634,0.009647,-0.135305,0.274111,-0.010271,-0.055769,...,0.262805,-0.072939,0.223894,0.081897,0.063673,-0.047841,0.099216,-0.003506,Durmiendo con su enemigo,3
3,0.038538,-0.024892,0.088787,0.257778,-0.110085,0.017308,-0.123375,0.300605,0.000616,-0.032242,...,0.276420,-0.086910,0.202238,0.088911,0.088554,-0.060159,0.109176,0.032100,Elizabethtown,4
4,0.041123,-0.059851,0.130128,0.262925,-0.114963,0.031149,-0.133662,0.248894,-0.001590,0.017231,...,0.319275,-0.089438,0.203360,0.062950,0.118264,-0.042241,0.128754,0.039183,Godzilla,5


## Embeddings de usuarios

Limpiamos las queries con el mismo proceso de antes

In [38]:
data_clean_users = pd.DataFrame(usuarios["query"].apply(limpieza))

### Embedding de las queries

In [39]:
embeddings_query = pd.DataFrame()

for query in data_clean_users["query"]:
    words = query.split()
    words_embeddings = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    embedding_mean = np.mean(words_embeddings, axis=0)
    embeddings_query = pd.concat([embeddings_query, pd.DataFrame(embedding_mean).T])

embeddings_query.reset_index(drop=True,inplace=True)

In [40]:
embeddings_query = embeddings_query.merge(usuarios[["id"]], left_index=True, right_index=True)
embeddings_query

,0,1,2,3,4,5,6,7,8,9,...,291,292,293,294,295,296,297,298,299,id
0,0.026638,-0.043014,0.089387,0.298044,-0.150599,0.011567,-0.108659,0.279608,-0.034860,-0.030066,...,0.172713,0.274478,-0.104489,0.206543,0.085951,0.108541,-0.046493,0.118209,0.013574,U01
1,0.020540,-0.045357,0.110235,0.254967,-0.110586,0.046356,-0.119272,0.277498,-0.008158,-0.018593,...,0.152580,0.275906,-0.101341,0.198007,0.083807,0.100805,-0.057594,0.121585,0.030757,U02
2,0.025239,-0.056122,0.096203,0.304331,-0.141284,0.017755,-0.132311,0.294974,-0.029244,-0.026521,...,0.175733,0.265837,-0.108570,0.220532,0.083839,0.104593,-0.053227,0.127539,0.014082,U03
3,0.045209,-0.070472,0.124774,0.316931,-0.149615,0.031683,-0.140347,0.278006,-0.044042,-0.029988,...,0.213167,0.283467,-0.103827,0.230559,0.089922,0.106492,-0.034283,0.166974,0.022295,U04
4,0.050085,-0.035493,0.142788,0.274009,-0.121322,0.033254,-0.131494,0.281415,-0.015857,-0.004120,...,0.172787,0.284828,-0.095683,0.202149,0.093694,0.112897,-0.045099,0.149009,0.038410,U05
5,0.015003,-0.087986,0.112452,0.285764,-0.148781,0.039906,-0.139026,0.289067,-0.023108,-0.041842,...,0.203203,0.298566,-0.110827,0.216981,0.059985,0.116997,-0.037203,0.154984,0.022880,U06
6,0.037464,-0.045579,0.107234,0.316416,-0.139250,0.022058,-0.136512,0.286126,-0.043088,-0.011083,...,0.185883,0.254580,-0.099102,0.225227,0.095296,0.105488,-0.046944,0.128531,0.013939,U07
7,0.038725,-0.066961,0.133728,0.261812,-0.128737,0.035362,-0.129264,0.266729,-0.007349,-0.015216,...,0.166061,0.318863,-0.099919,0.202826,0.062065,0.113528,-0.040095,0.151242,0.039067,U08
8,0.022985,-0.085865,0.120887,0.325576,-0.158501,0.036115,-0.135822,0.289292,-0.045028,-0.031331,...,0.218884,0.279699,-0.121871,0.223769,0.081010,0.122987,-0.035999,0.155277,0.017002,U09
9,0.037188,-0.063306,0.099470,0.319263,-0.156403,0.016035,-0.128964,0.293761,-0.045284,-0.045504,...,0.210846,0.256861,-0.109818,0.227151,0.094266,0.101704,-0.034144,0.147266,0.009172,U10


### Embedding historial

In [62]:
def calcular_embedding_historial(usuario, pelis_embd):
    peliculas_usuario = usuario[['pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']].tolist()
    embeddings = []
    for pelicula in peliculas_usuario:
        # check pelicula in pelis_embd
        if pelicula not in pelis_embd['name'].values:
            print(f"Película '{pelicula}' no encontrada en el DataFrame de embeddings.")
            continue
        embedding = pelis_embd[pelis_embd['name'] == pelicula].iloc[0][:-2].to_numpy(dtype=np.float32)
        embeddings.append(embedding)
    historial_embedding = np.mean(embeddings, axis=0)
    return historial_embedding

In [63]:
historiales_embeddings = []
for index, usuario in usuarios.iterrows():
    historial_embedding = calcular_embedding_historial(usuario, pelis_embd)
    historial_embedding = np.append(historial_embedding, usuario['id'])
    
    historiales_embeddings.append(historial_embedding)

historiales_df = pd.DataFrame(historiales_embeddings)

Película 'Rec' no encontrada en el DataFrame de embeddings.
Película 'El secreto de sus ojos' no encontrada en el DataFrame de embeddings.
Película 'Amélie' no encontrada en el DataFrame de embeddings.
Película 'El exorcista' no encontrada en el DataFrame de embeddings.
Película 'Intocable' no encontrada en el DataFrame de embeddings.
Película 'Una mente brillante' no encontrada en el DataFrame de embeddings.
Película 'Kill Bill' no encontrada en el DataFrame de embeddings.
Película 'Mamma Mia!' no encontrada en el DataFrame de embeddings.
Película 'Paddington' no encontrada en el DataFrame de embeddings.


Podemos agregarlas ya que no tiene sentido que falten

In [ ]:
df_pelis.to_csv("pelis_embd.csv", index=False)



In [ ]:
embeddings_query_jose = embeddings_query[embeddings_query['id'] == jose['id']]
jose_promedio = np.mean([embeddings_query_jose.iloc[0][:-1].to_numpy(dtype=np.float32), historial_jose], axis=0)
jose_promedio

array([ 2.07595062e-02,  1.69602394e-01,  4.33057249e-02, -1.23935968e-01,
        1.57278582e-01, -8.35606083e-02,  7.35978782e-02,  1.16239160e-01,
        4.73516881e-02,  5.12003480e-03, -1.24026053e-02, -8.79660025e-02,
        4.77575883e-02,  1.08049653e-01, -1.57699585e-01, -1.32207215e-01,
        2.14662567e-01, -3.54450271e-02,  3.53505686e-02,  4.55073640e-02,
       -3.00550200e-02,  2.17229594e-04,  1.84994310e-01, -4.74146195e-02,
        1.75365746e-01, -1.02013245e-01, -1.08271256e-01,  1.25407711e-01,
       -9.81374923e-03, -9.54016112e-03,  1.42388195e-01, -1.87269244e-02,
       -1.02935284e-01,  1.25134036e-01, -2.22097412e-02, -1.27624767e-02,
        1.95628345e-01, -3.28973651e-01, -3.16463597e-02, -1.88383043e-01,
       -1.22352399e-01, -3.52056623e-02,  4.11990210e-02,  5.05440235e-02,
        7.77541101e-02,  5.24775535e-02, -4.94123474e-02,  1.92725006e-03,
       -1.02063604e-01,  1.85610324e-01,  8.43111277e-02, -1.09585784e-02,
       -1.58922702e-01,  

In [ ]:
similares = w2v_model.wv.most_similar(positive=[jose_promedio],topn=10)
similares

[('varias', 0.9908871054649353),
 ('estilo', 0.9905944466590881),
 ('infancia', 0.9897976517677307),
 ('salir', 0.9887130260467529),
 ('bella', 0.9886916279792786),
 ('corazón', 0.9886060953140259),
 ('extraña', 0.9885649085044861),
 ('ir', 0.9885553121566772),
 ('seis', 0.9882989525794983),
 ('habitación', 0.9882380366325378)]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Extract movie embeddings from pelis_embd (assuming the last two columns are 'name' and 'id')
movie_embeddings = pelis_embd.iloc[:, :-2].to_numpy(dtype=np.float32)

# Calculate cosine similarity between jose_promedio and all movie embeddings
# Reshape jose_promedio to be 2D for cosine_similarity function
similarities = cosine_similarity(jose_promedio.reshape(1, -1), movie_embeddings)

# Get the indices of the top 10 most similar movies (descending order)
top_10_indices = similarities.argsort()[0][-10:][::-1]

# Get the corresponding movie names and their similarity scores
similar_movies_df = pd.DataFrame({
    'movie_name': pelis_embd.loc[top_10_indices, 'name'].values,
    'similarity_score': similarities[0, top_10_indices]
})

print("Top 10 Most Similar Movies for Jose:")
display(similar_movies_df)

Top 10 Most Similar Movies for Jose:


,movie_name,similarity_score
0,Más fuerte que su destino,0.998254
1,Todas contra él,0.998119
2,La angustia del miedo,0.998010
3,Un ángel en mi mesa,0.997948
4,El efecto mariposa,0.997947
5,Otoño en Nueva York,0.997826
6,Exorcismo en Connecticut,0.997742
7,Antes de amanecer,0.997740
8,Cuando cae la noche,0.997717
9,Conociendo a Matsuko,0.997674


## Opcion 1:
 Promedio del query con promedio de pelicula del historial contra promedio de pelicula.

## Opcion 2:
  Ponderar Promedio de query junto con el historial de pelicula, y compararlo con el promedio de pelicula.

## Opcion 3:
  Ponerle un peso al historial de peliculas por orden de visualizacion y compararlo con el promedio de pelicula.